# MS/MS Data Audit and Exploration

**Course:** DATA 266 - Fall 2026  
**Competition:** Enveda CASMI 2026 - Molecule ID From Mass Spectra  
**Task:** Data Processing and CNN Modeling

## Objectives

1. Load and inspect the competition dataset.
2. Understand the MS/MS spectral data structure.
3. Examine missing values and data quality.
4. Explore the distribution of molecular structures and spectra.
5. Identify preprocessing requirements for CNN models.

## 1. Download Dataset

Source: https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data

The dataset contains MS/MS spectra with associated molecular
structures represented as SMILES. Our goal is to prepare spectral
features for downstream deep learning models.

In [ ]:
!pip -q install -U kagglehub pyarrow polars

import os
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 30)

print("Environment setup completed!")

In [ ]:

kagglehub.login()

In [ ]:
COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"

data_dir = Path(
    kagglehub.competition_download(COMPETITION)
)

print("Dataset location:", data_dir)

for file in sorted(data_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(data_dir),
              f"{file.stat().st_size / 1024**2:.1f} MB")

In [ ]:
TRAIN_FILE = next(data_dir.rglob("train.parquet"))

pf = pq.ParquetFile(TRAIN_FILE)

print("Total rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)

print("\nDataset schema:")
for field in pf.schema_arrow:
    print(f"{field.name}: {field.type}")

In [ ]:
cols = [
    "ms2_mzs",
    "ms2_normalized_intensities",
    "precursor_mz",
    "adduct",
    "ionization_mode",
    "normalized_smiles",
    "inchikey14",
    "ingest_lib",
]

available_cols = [
    col for col in cols
    if col in pf.schema_arrow.names
]

batch = next(
    pf.iter_batches(
        batch_size=1000,
        columns=available_cols
    )
)

df_sample = batch.to_pandas()

print("Sample shape:", df_sample.shape)

display(
    df_sample.drop(
        columns=[
            "ms2_mzs",
            "ms2_normalized_intensities"
        ],
        errors="ignore"
    ).head()
)

print("\nMissing values in sample:")
print(df_sample.isna().sum())

In [ ]:
spectrum = df_sample.iloc[0]

mzs = np.asarray(
    spectrum["ms2_mzs"], dtype=float
)

intensities = np.asarray(
    spectrum["ms2_normalized_intensities"],
    dtype=float
)

assert len(mzs) == len(intensities), \
    "m/z and intensity arrays have different lengths"

valid = np.isfinite(mzs) & np.isfinite(intensities)

mzs = mzs[valid]
intensities = intensities[valid]

plt.figure(figsize=(12, 4))
plt.vlines(mzs, 0, intensities, linewidth=1)

plt.xlabel("m/z")
plt.ylabel("Normalized Intensity")
plt.title("Example MS/MS Spectrum")
plt.grid(alpha=0.2)
plt.show()

print("Number of peaks:", len(mzs))
print("Precursor m/z:", spectrum["precursor_mz"])

## 2. Dataset Diversity and Quality

The initial 1,000 records all come from the `drug_plus` library.
Therefore, the initial sample may not represent the full dataset.

We will examine samples from multiple Parquet row groups to
investigate:

- Distribution of source libraries
- Ionization modes and adduct types
- Number of spectral peaks
- Precursor m/z distribution
- Missing values across different data sources

This exploratory sample will help us design preprocessing
steps for the CNN models.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pf = pq.ParquetFile(TRAIN_FILE)

rng = np.random.default_rng(42)

eda_columns = [
    "ingest_lib",
    "inchikey14",
    "ionization_mode",
    "adduct",
    "precursor_mz",
    "num_peaks",
    "precursor_error_ppm"
]

samples = []

for i in range(pf.num_row_groups):

    table = pf.read_row_group(
        i,
        columns=eda_columns
    )

    df_group = table.to_pandas()

    n = min(300, len(df_group))

    selected_idx = rng.choice(
        len(df_group),
        size=n,
        replace=False
    )

    samples.append(
        df_group.iloc[selected_idx].copy()
    )

df_eda = pd.concat(
    samples,
    ignore_index=True
)

print("EDA sample shape:", df_eda.shape)

print("\nSource libraries:")
display(df_eda["ingest_lib"].value_counts())

print("\nIonization modes:")
display(df_eda["ionization_mode"].value_counts())

print("\nMissing values:")
display(df_eda.isna().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Source libraries
df_eda["ingest_lib"].value_counts().plot(
    kind="bar",
    ax=axes[0]
)
axes[0].set_title("Source Libraries")
axes[0].set_xlabel("Library")
axes[0].set_ylabel("Sample Count")

# Number of peaks
axes[1].hist(
    df_eda["num_peaks"].dropna(),
    bins=40
)
axes[1].set_title("Number of Peaks")
axes[1].set_xlabel("Peak Count")

# Precursor m/z
axes[2].hist(
    df_eda["precursor_mz"].dropna(),
    bins=40
)
axes[2].set_title("Precursor m/z Distribution")
axes[2].set_xlabel("Precursor m/z")

plt.tight_layout()
plt.show()

print("Peak count summary:")
display(df_eda["num_peaks"].describe())

print("Precursor m/z summary:")
display(df_eda["precursor_mz"].describe())

## 3. Exploratory Data Analysis: Key Findings

We examined 6,300 spectra sampled from 21 Parquet row
groups.

### Source Libraries
The dataset contains spectra from multiple libraries.
In our exploratory sample, enveda-180, pluskal_ms2,
and riken are the largest contributors.

Since we sampled an equal number of rows from each
row group, these proportions may not represent the
full dataset distribution.

### Ionization Modes
Approximately 74.3% of sampled spectra use positive
ionization mode, while 25.7% use negative mode.

Ionization mode and adduct information may be useful
additional features for the CNN models.

### Peak Count Distribution
The number of peaks varies considerably across spectra.

- Median: 44.5
- Mean: 157.2
- Maximum: 8,747

The distribution is strongly right-skewed.
Therefore, padding every spectrum to the maximum
peak count would be inefficient.

A fixed-length binned representation is a reasonable
initial approach for CNN input.

### Precursor m/z
The median precursor m/z is approximately 355.15,
with observed values ranging from 84.02 to 2001.84.

We need to examine fragment m/z distributions before
selecting the binning range and resolution.

### Missing Values
Only precursor_error_ppm contains missing values
(14 out of 6,300 sampled rows).

This field is related to labeled training structures
and should not be used as a prediction feature.

### Preliminary Conclusion
The EDA supports using spectrum binning to convert
variable-length MS/MS spectra into fixed-length CNN
inputs. Further analysis is required to choose the
bin width, m/z range, and preprocessing thresholds.

In [ ]:
# Sample spectral arrays from each Parquet row group

spectral_columns = [
    "ingest_lib",
    "ms2_mzs",
    "ms2_normalized_intensities",
    "precursor_mz",
    "num_peaks"
]

spectral_samples = []

for rg in range(pf.num_row_groups):

    batch = next(
        pf.iter_batches(
            batch_size=300,
            row_groups=[rg],
            columns=spectral_columns
        )
    )

    spectral_samples.append(batch.to_pandas())

df_spectra = pd.concat(
    spectral_samples,
    ignore_index=True
)

print("Sampled spectra:", len(df_spectra))

# Verify peak arrays
def check_spectrum(row):
    mz = np.asarray(row["ms2_mzs"], dtype=float)
    intensity = np.asarray(
        row["ms2_normalized_intensities"],
        dtype=float
    )

    return (
        len(mz) == len(intensity)
        and len(mz) > 0
        and np.all(np.isfinite(mz))
        and np.all(np.isfinite(intensity))
        and np.all(mz >= 0)
        and np.all(intensity >= 0)
    )

df_spectra["valid_spectrum"] = df_spectra.apply(
    check_spectrum,
    axis=1
)

print("Valid spectra:",
      df_spectra["valid_spectrum"].sum())

print("Invalid spectra:",
      (~df_spectra["valid_spectrum"]).sum())

In [ ]:
valid_spectra = df_spectra[
    df_spectra["valid_spectrum"]
]

all_mzs = np.concatenate(
    valid_spectra["ms2_mzs"].to_list()
)

print("Total sampled fragment peaks:", len(all_mzs))

print("\nFragment m/z quantiles:")
for q in [0, 0.25, 0.5, 0.75, 0.95, 0.99, 1]:
    print(
        f"{q*100:5.0f}th percentile:",
        round(np.quantile(all_mzs, q), 2)
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(all_mzs, bins=100)
axes[0].set_title("Fragment m/z Distribution")
axes[0].set_xlabel("Fragment m/z")
axes[0].set_ylabel("Peak Count")

axes[1].hist(
    all_mzs,
    bins=100,
    log=True
)
axes[1].set_title("Fragment m/z Distribution (Log Scale)")
axes[1].set_xlabel("Fragment m/z")
axes[1].set_ylabel("Peak Count (Log)")

plt.tight_layout()
plt.show()

## 4. Spectrum Binning Design

The fragment m/z distribution is strongly right-skewed.
Approximately 99% of sampled fragment peaks fall below
m/z 1337.36, while the maximum observed value is 2004.12.

For the initial CNN representation, we propose:

- m/z range: [0, 2200)
- Bin width: 1.0 Da
- Aggregation: Maximum intensity
- Intensity transformation: Square root
- Output dimension: 2200

These settings are preliminary and will be evaluated
against alternative binning resolutions.

We will also measure the information lost outside the
selected m/z range.

In [ ]:
mz_limits = [1000, 1500, 2000, 2200]

all_intensities = np.concatenate([
    np.asarray(x, dtype=np.float32)
    for x in valid_spectra["ms2_normalized_intensities"]
])

total_peaks = len(all_mzs)
total_intensity = all_intensities.sum()

results = []

for limit in mz_limits:

    mask = all_mzs >= limit

    results.append({
        "mz_limit": limit,
        "excluded_peaks": int(mask.sum()),
        "excluded_peak_pct": 100 * mask.mean(),
        "excluded_intensity_pct": (
            100 * all_intensities[mask].sum()
            / total_intensity
        )
    })

df_coverage = pd.DataFrame(results)

display(df_coverage)

In [ ]:
def spectrum_to_bins(
    mzs,
    intensities,
    mz_max=2200,
    bin_width=1.0
):

    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(
        intensities, dtype=np.float32
    )

    if len(mzs) != len(intensities):
        raise ValueError("m/z and intensity lengths differ.")

    n_bins = int(np.ceil(mz_max / bin_width))

    vector = np.zeros(
        n_bins, dtype=np.float32
    )

    # Remove invalid or out-of-range peaks
    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs >= 0)
        & (mzs < mz_max)
        & (intensities >= 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    # Square-root intensity transformation
    intensities = np.sqrt(intensities)

    # Map m/z values into bins
    bin_indices = np.floor(
        mzs / bin_width
    ).astype(np.int64)

    # Keep maximum intensity per bin
    np.maximum.at(
        vector,
        bin_indices,
        intensities
    )

    return vector

In [ ]:
sample_row = valid_spectra.iloc[0]

binned = spectrum_to_bins(
    sample_row["ms2_mzs"],
    sample_row["ms2_normalized_intensities"],
    mz_max=2200,
    bin_width=1.0
)

print("Binned shape:", binned.shape)
print("Non-zero bins:", np.count_nonzero(binned))
print("Max intensity:", binned.max())

assert binned.shape == (2200,)
assert np.all(np.isfinite(binned))
assert binned.min() >= 0

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Original spectrum
axes[0].vlines(
    sample_row["ms2_mzs"],
    0,
    sample_row["ms2_normalized_intensities"]
)

axes[0].set_title("Original Spectrum")
axes[0].set_xlabel("m/z")
axes[0].set_ylabel("Intensity")

# Binned spectrum
axes[1].plot(
    np.arange(len(binned)),
    binned
)

axes[1].set_title("Binned Spectrum (1 Da)")
axes[1].set_xlabel("m/z Bin")
axes[1].set_ylabel("Transformed Intensity")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    np.arange(len(binned)),
    binned,
    linewidth=1
)

plt.xlim(0, 400)
plt.xlabel("m/z")
plt.ylabel("Transformed Intensity")
plt.title("Binned MS/MS Spectrum (Zoomed)")
plt.grid(alpha=0.2)
plt.show()

## 5. Spectrum Binning Results

We converted a variable-length MS/MS spectrum into a
fixed-length representation using 1 Da binning.

### Results

- Input spectrum: 52 peaks
- Output vector dimension: 2200
- Non-zero bins: 48
- Maximum transformed intensity: 1.0

The number of non-zero bins is smaller than the
original peak count because multiple peaks can map
to the same bin, and zero-intensity peaks do not
produce non-zero values.

The square-root transformation increases the relative
contribution of low-intensity peaks.

### Conclusion

The initial binning implementation successfully
converts MS/MS spectra into fixed-length vectors
suitable for 1D CNN models.

The proposed binning range and resolution will be
evaluated during model experiments.